In [2]:
#imports 
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [11]:
# ⚙️ DEVICE CONFIG
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔧 HYPERPARAMETERS
LEARNING_RATE = 0.03  # learning rate
BATCH_SIZE = 64 # batch size
EPOCHS = 600 # number of epochs
HIDDEN_LAYERS = 2 # number of hidden layers
TRAIN_RATIO = 0.6 # ratio of training data
NUM_CLASSES = 2 # number of classes
NOISE = 1.0  # std deviation for clusters

In [12]:
import plotly.express as px
from sklearn.datasets import make_blobs
import pandas as pd

# Generate 3D blobs
X, y = make_blobs(n_samples=10000, centers=NUM_CLASSES, n_features=3,
                  cluster_std=NOISE, random_state=42)

# Convert to DataFrame for Plotly
df = pd.DataFrame(X, columns=["X1", "X2", "Z"])
df["Label"] = y

# Interactive Plot
fig = px.scatter_3d(df, x="X1", y="X2", z="Z", color="Label", opacity=0.6,
                    title="Interactive 3D Blob Dataset",
                    color_continuous_scale="Viridis")
fig.show()


In [6]:
X.shape

(10000, 3)

In [7]:
# 🟠 DATA PREPROCESSING
# Correct way for NumPy -> Torch
X = torch.tensor(X, dtype=torch.float32)  # shape [N, D]
y = torch.tensor(y, dtype=torch.long)     # shape [N]


# 🟠 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=TRAIN_RATIO, random_state=42)

# DATALOADERS
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)

In [8]:
# 🧠 MODEL
class MultiClassClassifier(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=16, hidden_layers=HIDDEN_LAYERS, output_dim=NUM_CLASSES):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers += [nn.Linear(hidden_dim, output_dim)]
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

model = MultiClassClassifier().to(device)
loss_fn = nn.CrossEntropyLoss()  # multi-class loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [9]:
# 🚂 TRAINING LOOP
for epoch in range(EPOCHS):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        y_logits = model(X_batch)
        loss = loss_fn(y_logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

Epoch 000 | Loss: 0.2714
Epoch 050 | Loss: 0.0626
Epoch 100 | Loss: 0.1487
Epoch 150 | Loss: 0.0738
Epoch 200 | Loss: 0.1198
Epoch 250 | Loss: 0.1713
Epoch 300 | Loss: 0.0804
Epoch 350 | Loss: 0.0776
Epoch 400 | Loss: 0.0335
Epoch 450 | Loss: 0.0959
Epoch 500 | Loss: 0.2085
Epoch 550 | Loss: 0.0580


In [10]:
# ✅ EVALUATION
model.eval()
with torch.no_grad():
    X_test, y_test = X_test.to(device), y_test.to(device)
    y_logits = model(X_test)
    y_pred = torch.argmax(y_logits, dim=1)
    acc = (y_pred == y_test).float().mean()
    print(f"\n🎯 Test Accuracy: {acc:.4f}")


🎯 Test Accuracy: 0.9538


In [18]:
import plotly.graph_objects as go

def plot_decision_boundary_3d_points(model, X, y, device):
    # Predict on full input
    model.eval()
    with torch.no_grad():
        X_device = X.to(device)
        preds = model(X_device)
        preds = torch.argmax(preds, dim=1)

    # Plot predicted points colored by model output
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=X[:, 0].cpu().numpy(),
        y=X[:, 1].cpu().numpy(),
        z=X[:, 2].cpu().numpy(),
        mode='markers',
        marker=dict(
            size=4,
            color=preds.cpu().numpy(),  # model-predicted class
            colorscale='Viridis',
            opacity=0.8
        ),
        name="Model Predictions"
    ))

    # Optionally overlay actual labels for comparison
    fig.add_trace(go.Scatter3d(
        x=X[:, 0].cpu().numpy(),
        y=X[:, 1].cpu().numpy(),
        z=X[:, 2].cpu().numpy(),
        mode='markers',
        marker=dict(
            size=2,
            color=y.cpu().numpy(),  # ground truth
            colorscale='curl',
            opacity=0.95,
            symbol="square-open"
        ),
        name="True Labels"
    ))

    fig.update_layout(
        title="3D Model Predictions (Colored Points)",
        scene=dict(
            xaxis_title='X1',
            yaxis_title='X2',
            zaxis_title='Z',
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()


In [19]:
plot_decision_boundary_3d_points(model, X_test, y_test, device)